# Jersey Number Recognition Pipeline

Runs the full jersey number recognition pipeline on the SoccerNet jersey-2023 dataset.

**Before running:** `Runtime → Change runtime type → GPU (T4)`

## Pipeline stages
`soccer_ball_filter → reid features → gaussian filter → legibility → pose (ViTPose) → crops → STR (PARSeq) → combine → eval`


## Step 1 — Check GPU

In [4]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type and select GPU.")

print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"CUDA version:   {torch.version.cuda}")
print(f"PyTorch version:{torch.__version__}")


GPU:            Tesla T4
CUDA version:   12.8
PyTorch version:2.10.0+cu128


## Optional — Reset (clean slate)

Run **only** if you want to wipe everything and start fresh.  
Does NOT reset pip packages — use `Runtime → Disconnect and delete runtime` for that.


In [5]:
import shutil, os

DIRS_TO_DELETE = [
    "/content/jersey-pipeline",
    "/content/mmcv-src",
]
for d in DIRS_TO_DELETE:
    if os.path.isdir(d):
        shutil.rmtree(d)
        print(f"🗑️  Deleted {d}")
    else:
        print(f"⏭️  Skipped {d} (not found)")

os.chdir('/content')
print("\n✅ Reset complete — re-run all cells from Step 2 onwards.")


🗑️  Deleted /content/jersey-pipeline
⏭️  Skipped /content/mmcv-src (not found)

✅ Reset complete — re-run all cells from Step 2 onwards.


## Step 2 — Clone the repository

In [6]:
import os
from google.colab import userdata

# ── CONFIG ───────────────────────────────────────────────────────────────────
# Repo is private. Add your GitHub PAT to Colab Secrets (🔑 sidebar):
#   Name: GITHUB_TOKEN  |  Value: your token (needs 'repo' scope)
# ─────────────────────────────────────────────────────────────────────────────
REPO_OWNER  = "Li11z"
REPO_NAME   = "419-deeplearning-project"
REPO_BRANCH = "jersey_solution"
REPO_DIR    = "/content/jersey-pipeline"

os.system('git config --global user.email "colab@example.com"')
os.system('git config --global user.name "Colab User"')

try:
    TOKEN = userdata.get('GITHUB_TOKEN')
    REPO_URL = f"https://{TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    print("Using GitHub token from Colab Secrets.")
except Exception:
    REPO_URL = f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git"
    print("No GITHUB_TOKEN — attempting public clone.")

os.chdir('/content')

if os.path.isdir(REPO_DIR):
    print(f"Repo already exists at {REPO_DIR}, skipping clone.")
else:
    # os.system(f"git clone --depth 1 {REPO_URL} {REPO_DIR}")
    os.system(f"git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
os.system(f"git checkout {REPO_BRANCH}")
print("Working directory:", os.getcwd())


Using GitHub token from Colab Secrets.
Working directory: /content/jersey-pipeline


## Step 3 — Install dependencies

> **Note on mmcv:** `pose.py` imports `mmpose.apis` which is part of ViTPose and requires
> `mmcv-full` (with compiled CUDA ops). There is no prebuilt wheel for CUDA 12.8 / Python 3.12,
> so we build it from source. **This takes ~15 minutes — do not interrupt.**


In [7]:
# 3a. Core pipeline dependencies
!pip install -q \
    numpy==1.26.4 \
    pandas tqdm scipy \
    opencv-python gdown \
    SoccerNet yacs einops \
    pycocotools \
    pytorch_lightning \
    timm==0.4.9 \
    hydra-core \
    lmdb imgaug tensorboardx omegaconf \
    packaging

print("✅ Core deps installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.1/346.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [8]:
# 3b. Build mmcv-full v1.7.2 from source
# Required by ViTPose's mmpose.apis — no prebuilt wheel for CUDA 12.8 / Python 3.12
# ~15 minutes. Do not interrupt.
import os, sys

print(f"Python {sys.version_info.major}.{sys.version_info.minor} | CUDA {__import__('torch').version.cuda}")
print("Building mmcv-full from source — this will take ~15 minutes...")

os.chdir('/content')
!pip install -q ninja cmake

if not os.path.isdir('/content/mmcv-src'):
    !git clone -b v1.7.2 --depth 1 https://github.com/open-mmlab/mmcv.git mmcv-src

os.chdir('/content/mmcv-src')

# Fix 1: upgrade setuptools so pkg_resources works on Python 3.12
!pip install -q -U setuptools packaging

# Fix 2: patch setup.py — pkg_resources.parse_version is broken on Python 3.12
with open('setup.py') as f:
    src = f.read()
if 'from pkg_resources import' in src:
    src = src.replace(
        'from pkg_resources import DistributionNotFound, get_distribution, parse_version',
        'from packaging.version import Version as parse_version\n'
        'def get_distribution(name):\n'
        '    import importlib.metadata\n'
        '    class D:\n'
        '        version = importlib.metadata.version(name)\n'
        '    return D()\n'
        'DistributionNotFound = Exception'
    )
    with open('setup.py', 'w') as f:
        f.write(src)
    print("setup.py patched.")

# Fix 3: patch distutils imports (removed in Python 3.12)
!grep -rl 'from distutils' mmcv/ | xargs sed -i \
    's/from distutils/from setuptools._distutils/g' 2>/dev/null || true

# Build with CUDA ops
!MMCV_WITH_OPS=1 pip install -e . 2>&1 | tail -15

# Editable installs don't auto-register — add to sys.path manually
import sys, glob
sys.path.insert(0, '/content/mmcv-src')
if 'mmcv' in sys.modules:
    del sys.modules['mmcv']

os.chdir('/content/jersey-pipeline')
import mmcv
print(f"✅ mmcv {mmcv.__version__} ready.")


Python 3.12 | CUDA 12.8
Building mmcv-full from source — this will take ~15 minutes...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 5.9 MB/s eta 0:00:00
Cloning into 'mmcv-src'...
remote: Enumerating objects: 1065, done.
remote: Counting objects: 100% (1065/1065), done.
remote: Compressing objects: 100% (945/945), done.
remote: Total 1065 (delta 178), reused 438 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (1065/1065), 6.47 MiB | 14.54 MiB/s, done.
Resolving deltas: 100% (178/178), done.
Note: switching to '4c01b026f0afa5a91a5f54aea313788da1e40f95'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation 

/content/mmcv-src/mmcv/__init__.py:20: UserWarning: On January 1, 2023, MMCV will release v2.0.0, in which it will remove components related to the training process and add a data transformation module. In addition, it will rename the package names mmcv to mmcv-lite and mmcv-full to mmcv. See https://github.com/open-mmlab/mmcv/blob/master/docs/en/compatibility.md for more details.
  warnings.warn(


In [9]:
# 3c. Install ViTPose (provides mmpose.apis used by pose.py)
import os, sys
sys.path.insert(0, '/content/mmcv-src')  # ensure mmcv is findable during install

VITPOSE_DIR = "/content/jersey-pipeline/pose/ViTPose"

if not os.path.isdir(VITPOSE_DIR):
    os.chdir('/content/jersey-pipeline')
    !git clone --recurse-submodules \
        https://github.com/ViTAE-Transformer/ViTPose.git {VITPOSE_DIR}

os.chdir(VITPOSE_DIR)
!pip install -q -e . 2>&1 | tail -5
!pip install -q timm==0.4.9 einops

os.chdir('/content/jersey-pipeline')
print("✅ ViTPose installed.")


Cloning into '/content/jersey-pipeline/pose/ViTPose'...
remote: Enumerating objects: 1868, done.
remote: Counting objects: 100% (1209/1209), done.
remote: Compressing objects: 100% (422/422), done.
remote: Total 1868 (delta 860), reused 787 (delta 787), pack-reused 659 (from 1)
Receiving objects: 100% (1868/1868), 10.75 MiB | 21.22 MiB/s, done.
Resolving deltas: 100% (962/962), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 3.2 MB/s eta 0:00:00
✅ ViTPose installed.


In [10]:
# 3d. Install PARSeq deps (bundled in str/parseq — no clone needed)
import os

PARSEQ_DIR = "/content/jersey-pipeline/str/parseq"
os.chdir(PARSEQ_DIR)

# Install without pinned torch/torchvision (Colab already has compatible versions)
!pip install -q -e ".[train,test]" --no-deps 2>&1 | tail -5

os.chdir('/content/jersey-pipeline')
print("✅ PARSeq installed.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/jersey-pipeline/str/parseq'

In [ ]:
# 3e. Install Centroids-ReID deps (bundled in reid/centroids-reid — no clone needed)
# Skips requirements.txt which pins torch==1.7.1+cu101 (long obsolete)
import os

REID_DIR = "/content/jersey-pipeline/reid/centroids-reid"

if not os.path.isdir(REID_DIR):
    os.chdir('/content/jersey-pipeline')
    !git clone --recurse-submodules \
        https://github.com/mikwieczorek/centroids-reid.git {REID_DIR}

!pip install -q yacs faiss-gpu pytorch-ignite 2>&1 | tail -5

os.chdir('/content/jersey-pipeline')
print("✅ Centroids-ReID deps installed.")


In [ ]:
# 3f. Verify all critical imports work
import sys
sys.path.insert(0, '/content/mmcv-src')
sys.path.insert(0, '/content/jersey-pipeline')
sys.path.insert(0, '/content/jersey-pipeline/pose/ViTPose')
sys.path.insert(0, '/content/jersey-pipeline/str/parseq')
sys.path.insert(0, '/content/jersey-pipeline/reid/centroids-reid')

import mmcv;         print(f"✅ mmcv          {mmcv.__version__}")
import torch;        print(f"✅ torch         {torch.__version__}")
import cv2;          print(f"✅ opencv        {cv2.__version__}")

try:
    from mmpose.apis import init_pose_model
    print("✅ mmpose.apis   ok")
except Exception as e:
    print(f"❌ mmpose.apis   {e}")

try:
    from strhub.models.utils import load_from_checkpoint
    print("✅ strhub        ok")
except Exception as e:
    print(f"❌ strhub        {e}")

try:
    from train_ctl_model import CTLModel
    print("✅ centroids-reid ok")
except Exception as e:
    print(f"❌ centroids-reid {e}")


## Step 4 — Download pretrained models

Downloads:
- **ViTPose-H** (~600 MB) — pose estimation
- **Legibility classifier** (ResNet-34)
- **PARSeq** — jersey number STR model
- **ReID** — market1501 ResNet-50 weights


In [ ]:
import gdown, os, sys

PIPELINE_DIR = "/content/jersey-pipeline"
sys.path.insert(0, PIPELINE_DIR)
os.chdir(PIPELINE_DIR)
import configuration as cfg

os.makedirs(f"{PIPELINE_DIR}/models", exist_ok=True)
os.makedirs(f"{PIPELINE_DIR}/pose/ViTPose/checkpoints", exist_ok=True)

# ViTPose-H checkpoint
vitpose_ckpt = f"{PIPELINE_DIR}/pose/ViTPose/checkpoints/vitpose-h.pth"
if not os.path.isfile(vitpose_ckpt):
    print("Downloading ViTPose-H (~600 MB)...")
    gdown.download(cfg.dataset['SoccerNet']['pose_model_url'], vitpose_ckpt, quiet=False)
else:
    print("✅ ViTPose-H already downloaded.")

# Legibility model
leg_model = f"{PIPELINE_DIR}/{cfg.dataset['SoccerNet']['legibility_model']}"
if not os.path.isfile(leg_model):
    print("Downloading legibility model...")
    gdown.download(cfg.dataset['SoccerNet']['legibility_model_url'], leg_model, quiet=False)
else:
    print("✅ Legibility model already downloaded.")

# STR model
str_model = f"{PIPELINE_DIR}/{cfg.dataset['SoccerNet']['str_model']}"
if not os.path.isfile(str_model):
    print("Downloading STR model...")
    gdown.download(cfg.dataset['SoccerNet']['str_model_url'], str_model, quiet=False)
else:
    print("✅ STR model already downloaded.")

# ReID weights
reid_ckpt = f"{PIPELINE_DIR}/reid/centroids-reid/models/market1501_resnet50_256_128_epoch_120.ckpt"
os.makedirs(os.path.dirname(reid_ckpt), exist_ok=True)
if not os.path.isfile(reid_ckpt):
    print("Downloading ReID weights...")
    gdown.download(
        "https://drive.google.com/uc?id=1ZFywKEytpyNocUQd2APh2XqTe8X0HMom",
        reid_ckpt, quiet=False
    )
else:
    print("✅ ReID weights already downloaded.")

print("\n✅ All models ready.")


## Step 5 — Download the SoccerNet jersey-2023 dataset

You need a [SoccerNet](https://www.soccer-net.org/) account with NDA-accepted credentials.

Add them to Colab Secrets (🔑 key icon in sidebar):
- `SOCCERNET_USER` ← your registered email  
- `SOCCERNET_PASS` ← your password


In [ ]:
from google.colab import userdata

try:
    SOCCERNET_USER = userdata.get('SOCCERNET_USER')
    SOCCERNET_PASS = userdata.get('SOCCERNET_PASS')
    print("✅ Credentials loaded from Colab Secrets.")
except Exception:
    SOCCERNET_USER = "your_email@example.com"  # ← edit here if not using Secrets
    SOCCERNET_PASS = "your_password"
    print("⚠️  No secrets found — using hardcoded credentials.")


In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
import zipfile, os

PIPELINE_DIR = "/content/jersey-pipeline"
os.chdir(PIPELINE_DIR)

RAW_DIR = f"{PIPELINE_DIR}/Soccer_Jersey_Dataset"
os.makedirs(RAW_DIR, exist_ok=True)

mySN = SoccerNetDownloader(LocalDirectory=RAW_DIR)
mySN.username = SOCCERNET_USER
mySN.password = SOCCERNET_PASS
mySN.downloadDataTask(task="jersey-2023", split=["train", "test", "challenge"])

# Extract archives
jersey_dir = os.path.join(RAW_DIR, "jersey-2023")
for split in ["train", "test", "challenge"]:
    zip_path = os.path.join(jersey_dir, f"{split}.zip")
    dest = os.path.join(PIPELINE_DIR, "data", split)
    if os.path.isfile(zip_path) and not os.path.isdir(dest):
        print(f"Extracting {split}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(os.path.join(PIPELINE_DIR, "data"))
    elif os.path.isdir(dest):
        print(f"✅ {split} already extracted.")

print("✅ Dataset ready in data/")


## Step 6 — Run inference

| Stage | Script | Description |
|---|---|---|
| `soccer_ball_filter` | helpers.py | Remove soccer-ball tracklets by image size |
| `feat` | centroid_reid.py | Extract ReID features (GPU) |
| `filter` | gaussian_outliers.py | Gaussian outlier removal |
| `legible` | legibility_classifier.py | Legibility classification (GPU) |
| `pose` | pose.py + ViTPose | Pose estimation (GPU) |
| `crops` | helpers.py | Torso crop generation |
| `str` | str.py + PARSeq | Jersey number recognition (GPU) |
| `combine` | helpers.py | Aggregate tracklet predictions |
| `eval` | helpers.py | Compute accuracy vs ground truth |


In [ ]:
import torch, os, sys

assert torch.cuda.is_available(), "GPU lost — re-run Step 1"

# Ensure all sub-repos are on the path for subprocess calls via main.py
PIPELINE_DIR = "/content/jersey-pipeline"
sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, '/content/mmcv-src')
sys.path.insert(0, f"{PIPELINE_DIR}/pose/ViTPose")
sys.path.insert(0, f"{PIPELINE_DIR}/str/parseq")
sys.path.insert(0, f"{PIPELINE_DIR}/reid/centroids-reid")

os.chdir(PIPELINE_DIR)
print(f"✅ Running on: {torch.cuda.get_device_name(0)}")


In [ ]:
# Run full SoccerNet test pipeline
# main.py spawns subprocesses for each stage, so sys.path set above won't carry over.
# We use PYTHONPATH to ensure all sub-repos are findable in child processes.
import os

PIPELINE_DIR = "/content/jersey-pipeline"
PYTHONPATH = ":".join([
    "/content/mmcv-src",
    f"{PIPELINE_DIR}/pose/ViTPose",
    f"{PIPELINE_DIR}/str/parseq",
    f"{PIPELINE_DIR}/reid/centroids-reid",
    PIPELINE_DIR,
])

os.chdir(PIPELINE_DIR)
os.environ['PYTHONPATH'] = PYTHONPATH
!PYTHONPATH={PYTHONPATH} python main.py SoccerNet test


Results are saved to `out/SoccerNetResults/`.

---
## Optional — Run on validation split

In [ ]:
import os
PIPELINE_DIR = "/content/jersey-pipeline"
PYTHONPATH = ":".join([
    "/content/mmcv-src",
    f"{PIPELINE_DIR}/pose/ViTPose",
    f"{PIPELINE_DIR}/str/parseq",
    f"{PIPELINE_DIR}/reid/centroids-reid",
    PIPELINE_DIR,
])
os.chdir(PIPELINE_DIR)
!PYTHONPATH={PYTHONPATH} python main.py SoccerNet val


---
## Optional — Run individual pipeline stages

Set `True` only for the stages you want to run.

In [ ]:
import sys, argparse, os
PIPELINE_DIR = "/content/jersey-pipeline"
sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, '/content/mmcv-src')
sys.path.insert(0, f"{PIPELINE_DIR}/pose/ViTPose")
sys.path.insert(0, f"{PIPELINE_DIR}/str/parseq")
sys.path.insert(0, f"{PIPELINE_DIR}/reid/centroids-reid")
os.chdir(PIPELINE_DIR)

import main as pipeline

args = argparse.Namespace()
args.dataset = 'SoccerNet'
args.part    = 'test'
args.pipeline = {
    "soccer_ball_filter": False,
    "feat":               False,
    "filter":             False,
    "legible":            False,
    "legible_eval":       False,
    "pose":               False,
    "crops":              False,
    "str":                False,
    "combine":            True,   # ← set True for stages you want to run
    "eval":               True,
}
pipeline.soccer_net_pipeline(args)


---
## Optional — Train PARSeq on SoccerNet data

In [ ]:
import os
PIPELINE_DIR = "/content/jersey-pipeline"
PYTHONPATH = ":".join([
    "/content/mmcv-src",
    f"{PIPELINE_DIR}/pose/ViTPose",
    f"{PIPELINE_DIR}/str/parseq",
    f"{PIPELINE_DIR}/reid/centroids-reid",
    PIPELINE_DIR,
])
os.chdir(PIPELINE_DIR)
!PYTHONPATH={PYTHONPATH} python main.py SoccerNet train --train_str
